In [1]:
# Repositorio : https://github.com/asier-ortiz/ai-course-notebooks/tree/main/actividad10

## Actividad 1

In [2]:
from pyspark.sql import SparkSession

In [3]:
# Crear SparkSession.
spark = SparkSession.builder \
    .appName("actividad_10") \
    .config("spark.ui.port", "4050") \
    .getOrCreate()

# Cargar el CSV cars.
df = spark.read.csv("../data/cars.csv", sep=";", header=True, inferSchema=True)

# Registrar la tabla como vista temporal para hacer las consultas SQL
df.createOrReplaceTempView("cars")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/03 11:51:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# Mostrar esquema de la tabla.
spark.sql("DESCRIBE cars").show(truncate=False)

+------------+------------+-------+
|col_name    |data_type   |comment|
+------------+------------+-------+
|Car         |string      |null   |
|MPG         |double      |null   |
|Cylinders   |int         |null   |
|Displacement|double      |null   |
|Horsepower  |double      |null   |
|Weight      |decimal(4,0)|null   |
|Acceleration|double      |null   |
|Model       |int         |null   |
|Origin      |string      |null   |
+------------+------------+-------+



In [5]:
# Mostrar las primeras 5 filas. 
spark.sql("SELECT * FROM cars LIMIT 5").show(truncate=False)

+-------------------------+----+---------+------------+----------+------+------------+-----+------+
|Car                      |MPG |Cylinders|Displacement|Horsepower|Weight|Acceleration|Model|Origin|
+-------------------------+----+---------+------------+----------+------+------------+-----+------+
|Chevrolet Chevelle Malibu|18.0|8        |307.0       |130.0     |3504  |12.0        |70   |US    |
|Buick Skylark 320        |15.0|8        |350.0       |165.0     |3693  |11.5        |70   |US    |
|Plymouth Satellite       |18.0|8        |318.0       |150.0     |3436  |11.0        |70   |US    |
|AMC Rebel SST            |16.0|8        |304.0       |150.0     |3433  |12.0        |70   |US    |
|Ford Torino              |17.0|8        |302.0       |140.0     |3449  |10.5        |70   |US    |
+-------------------------+----+---------+------------+----------+------+------------+-----+------+



In [6]:
# Mostrar las columnas «Cars» y «Cylinders» de todos los vehículos que sean de «Europa».
spark.sql("""
          SELECT Car, Cylinders
          FROM cars
          WHERE Origin = 'Europe'
          """).show(truncate=False)

+----------------------------+---------+
|Car                         |Cylinders|
+----------------------------+---------+
|Citroen DS-21 Pallas        |4        |
|Volkswagen 1131 Deluxe Sedan|4        |
|Peugeot 504                 |4        |
|Audi 100 LS                 |4        |
|Saab 99e                    |4        |
|BMW 2002                    |4        |
|Volkswagen Super Beetle 117 |4        |
|Opel 1900                   |4        |
|Peugeot 304                 |4        |
|Fiat 124B                   |4        |
|Volkswagen Model 111        |4        |
|Volkswagen Type 3           |4        |
|Volvo 145e (sw)             |4        |
|Volkswagen 411 (sw)         |4        |
|Peugeot 504 (sw)            |4        |
|Renault 12 (sw)             |4        |
|Volkswagen Super Beetle     |4        |
|Fiat 124 Sport Coupe        |4        |
|Fiat 128                    |4        |
|Opel Manta                  |4        |
+----------------------------+---------+
only showing top

In [7]:
# Obtener la media de «Horsepower», «Weight» y «Acceleration» por «Origen». 
spark.sql("""
          SELECT Origin,
                 ROUND(AVG(Horsepower), 2)   AS avg_hp,
                 ROUND(AVG(Weight), 2)       AS avg_weight,
                 ROUND(AVG(Acceleration), 2) AS avg_acc
          FROM cars
          GROUP BY Origin
          """).show()

+------+------+----------+-------+
|Origin|avg_hp|avg_weight|avg_acc|
+------+------+----------+-------+
|Europe| 78.78|   2431.49|  16.82|
|    US|118.01|   3372.70|  14.94|
| Japan| 79.84|   2221.23|  16.17|
+------+------+----------+-------+



In [8]:
# Calcular el ratio entre potencia y peso y, a continuación, sacar la media por cantidad de cilindros. 
spark.sql("""
          SELECT Cylinders,
                 ROUND(AVG(Horsepower / Weight), 4) AS avg_hp_weight_ratio
          FROM cars
          WHERE Weight IS NOT NULL
            AND Horsepower IS NOT NULL
          GROUP BY Cylinders
          """).show()

+---------+-------------------+
|Cylinders|avg_hp_weight_ratio|
+---------+-------------------+
|        6|             0.0316|
|        3|             0.0414|
|        5|              0.027|
|        4|             0.0332|
|        8|             0.0387|
+---------+-------------------+



## Actividad 2

In [9]:
# Cargar el CSV de las mediciones de enero de 2020.
df = spark.read.csv("../data/ene_mo20.csv", sep=";", header=True, inferSchema=True)

# Evita el warning por truncado del plan lógico al crear la vista SQL temporal.
spark.conf.set("spark.sql.debug.maxToStringFields", 1_000)

# Registrar la tabla como vista temporal para hacer las consultas SQL.
df.createOrReplaceTempView("air")

In [10]:
# Mostrar esquema de la tabla.
df.printSchema()

root
 |-- PROVINCIA: integer (nullable = true)
 |-- MUNICIPIO: integer (nullable = true)
 |-- ESTACION: integer (nullable = true)
 |-- MAGNITUD: integer (nullable = true)
 |-- PUNTO_MUESTREO: string (nullable = true)
 |-- ANO: integer (nullable = true)
 |-- MES: integer (nullable = true)
 |-- DIA: integer (nullable = true)
 |-- H01: double (nullable = true)
 |-- V01: string (nullable = true)
 |-- H02: double (nullable = true)
 |-- V02: string (nullable = true)
 |-- H03: double (nullable = true)
 |-- V03: string (nullable = true)
 |-- H04: double (nullable = true)
 |-- V04: string (nullable = true)
 |-- H05: double (nullable = true)
 |-- V05: string (nullable = true)
 |-- H06: double (nullable = true)
 |-- V06: string (nullable = true)
 |-- H07: double (nullable = true)
 |-- V07: string (nullable = true)
 |-- H08: double (nullable = true)
 |-- V08: string (nullable = true)
 |-- H09: double (nullable = true)
 |-- V09: string (nullable = true)
 |-- H10: double (nullable = true)
 |-- V10: 

In [11]:
# Mostrar el primer registro verticalemte.
spark.sql("SELECT * FROM air LIMIT 1").show(vertical=True)

-RECORD 0-----------------------
 PROVINCIA      | 28            
 MUNICIPIO      | 79            
 ESTACION       | 4             
 MAGNITUD       | 1             
 PUNTO_MUESTREO | 28079004_1_38 
 ANO            | 2020          
 MES            | 1             
 DIA            | 1             
 H01            | 7.0           
 V01            | V             
 H02            | 8.0           
 V02            | V             
 H03            | 9.0           
 V03            | V             
 H04            | 8.0           
 V04            | V             
 H05            | 6.0           
 V05            | V             
 H06            | 6.0           
 V06            | V             
 H07            | 5.0           
 V07            | V             
 H08            | 5.0           
 V08            | V             
 H09            | 4.0           
 V09            | V             
 H10            | 5.0           
 V10            | V             
 H11            | 6.0           
 V11      

In [12]:
# Indica el número de estaciones distintas que hay en los ficheros. 
spark.sql("""
          SELECT COUNT(DISTINCT ESTACION) AS num_stations
          FROM air
          """).show()

+------------+
|num_stations|
+------------+
|          24|
+------------+



In [13]:
# Indica el número de los distintos parámetros que se miden.
spark.sql("""
          SELECT COUNT(DISTINCT MAGNITUD) AS num_parameters
          FROM air
          """).show()

+--------------+
|num_parameters|
+--------------+
|            14|
+--------------+



In [14]:
# Indica el número de filas que hay para el día 18-01-2020. 
spark.sql("""
          SELECT COUNT(*) AS num_rows
          FROM air
          WHERE ANO = 2020
            AND MES = 1
            AND DIA = 18
          """).show()

+--------+
|num_rows|
+--------+
|     153|
+--------+



In [15]:
# Averigua la media de dióxido de azufre a las 12h de cada día. 
spark.sql("""
          SELECT ANO AS YEAR,
        MES AS MONTH, 
        DIA AS DAY, 
        ROUND(AVG(H12), 2) AS SO2_avg_12h
          FROM air
          WHERE MAGNITUD = 1 AND V12 = 'V'
          GROUP BY ANO, MES, DIA
          ORDER BY ANO, MES, DIA
          """).show(31, truncate=False)

+----+-----+---+-----------+
|YEAR|MONTH|DAY|SO2_avg_12h|
+----+-----+---+-----------+
|2020|1    |1  |9.5        |
|2020|1    |2  |11.2       |
|2020|1    |3  |8.6        |
|2020|1    |4  |5.4        |
|2020|1    |5  |6.0        |
|2020|1    |6  |7.0        |
|2020|1    |7  |11.1       |
|2020|1    |8  |12.4       |
|2020|1    |9  |12.8       |
|2020|1    |10 |6.5        |
|2020|1    |11 |5.6        |
|2020|1    |12 |8.0        |
|2020|1    |13 |9.0        |
|2020|1    |14 |8.2        |
|2020|1    |15 |8.1        |
|2020|1    |16 |8.2        |
|2020|1    |17 |5.9        |
|2020|1    |18 |5.5        |
|2020|1    |19 |4.5        |
|2020|1    |20 |4.6        |
|2020|1    |21 |5.11       |
|2020|1    |22 |6.22       |
|2020|1    |23 |7.44       |
|2020|1    |24 |7.0        |
|2020|1    |25 |5.75       |
|2020|1    |26 |6.56       |
|2020|1    |27 |6.67       |
|2020|1    |28 |6.25       |
|2020|1    |29 |6.13       |
|2020|1    |30 |5.44       |
|2020|1    |31 |6.0        |
+----+-----+--